In [ ]:
"""GF(2) Multiplication using FLASQ

This notebook demonstrates resource estimation for two different GF(2)
multiplication circuits using the FLASQ model. We compare a standard
'schoolbook' quadratic multiplication algorithm with a recursive Karatsuba-based
algorithm.

The circuits are constructed using builder functions that replicate a specific
2D qubit layout on a grid, which is then analyzed using the modern FLASQ
costing tools.
"""

In a real environment, you would install qualtran and qualtran-flasq.
For this example, we assume they are available in the path.

In [ ]:
from frozendict import frozendict

In [ ]:
from qualtran.resource_counting import get_cost_value, QubitCount
from qualtran.surface_code.flasq.cirq_interop import convert_circuit_for_flasq_analysis
from qualtran.surface_code.flasq.examples.gf2_multiplier import (
    build_karatsuba_mult_circuit,
    build_quadratic_mult_circuit,
)
from qualtran.surface_code.flasq.flasq_model import (
    apply_flasq_cost_model,
    conservative_FLASQ_costs,
    optimistic_FLASQ_costs,
    FLASQSummary,
    FLASQCostModel,
)
from qualtran.surface_code.flasq.measurement_depth import TotalMeasurementDepth
from qualtran.surface_code.flasq.span_counting import TotalSpanCost
from qualtran.surface_code.flasq.symbols import ROTATION_ERROR, T_REACT, V_CULT_FACTOR
from qualtran.surface_code.flasq.volume_counting import FLASQGateTotals

In [ ]:
from qualtran import CompositeBloq, Bloq

pylint: disable=line-too-long

In [ ]:
import os

In [ ]:
IS_FAST_MODE = os.environ.get("FLASQ_FAST_MODE_OVERRIDE", "False").lower() == "true"

In [ ]:
# --- Analysis Parameters ---
if IS_FAST_MODE:
    BITSIZE = 5
else:
    BITSIZE = 10

In [ ]:
print(f"--- Analyzing GF(2) Multiplication for bitsize = {BITSIZE} ---")

In [ ]:
def run_and_print_flasq_analysis(
    title: str,
    cbloq: "CompositeBloq",
    n_total_logical_qubits: int,
    cost_model: "FLASQCostModel",
    model_name: str,
    assumptions: "frozendict",
) -> "FLASQSummary":
    """Runs the FLASQ analysis for a given bloq and prints a summary."""
    print(f"\n{title} ({model_name} Model):")

    # Get resource counts from the bloq
    qubits = get_cost_value(cbloq, QubitCount())
    depth = get_cost_value(cbloq, TotalMeasurementDepth())
    totals = get_cost_value(cbloq, FLASQGateTotals())
    span = get_cost_value(cbloq, TotalSpanCost())

    # Apply the FLASQ model
    summary_symbolic = apply_flasq_cost_model(
        model=cost_model,
        n_total_logical_qubits=n_total_logical_qubits,
        qubit_counts=qubits,
        counts=totals,
        span_info=span,
        measurement_depth=depth,
        logical_timesteps_per_measurement=1,  # Example value
    )

    # Resolve symbols
    summary_resolved = summary_symbolic.resolve_symbols(assumptions=assumptions)

    # Print summary
    print(f"    Total Logical Qubit Budget: {n_total_logical_qubits}")
    print(f"    Number of Algorithmic Qubits: {summary_resolved.n_algorithmic_qubits}")
    print(f"    Number of Fluid Ancillas: {summary_resolved.n_fluid_ancilla}")
    print(f"    Total T-gates: {summary_resolved.total_t_count:,.0f}")
    print(
        f"    Total Computational Volume: {summary_resolved.total_computational_volume:,.2f}"
    )
    print(f"    Total Spacetime Volume: {summary_resolved.total_spacetime_volume:,.2f}")
    print(f"    Total Depth (Logical Timesteps): {summary_resolved.total_depth:,.2f}")
    print(
        f"    Regime: {'Volume-limited' if summary_resolved.is_volume_limited else 'Reaction-limited'}"
    )
    return summary_resolved

In [ ]:
# --- 1. Schoolbook (Quadratic) Multiplication ---
print("\n1. Analyzing Schoolbook (Quadratic) Multiplier...")
# Build the circuit with the specific 2D layout.
qmult_data = build_quadratic_mult_circuit(bitsize=BITSIZE)

In [ ]:
# Convert the circuit to a CompositeBloq for costing.
qmult_cbloq, _ = convert_circuit_for_flasq_analysis(qmult_data.circuit)

In [ ]:
# Get the abstract gate counts and span costs.
qmult_totals = get_cost_value(qmult_cbloq, FLASQGateTotals())
qmult_span = get_cost_value(qmult_cbloq, TotalSpanCost())

In [ ]:
print("  Resource Counts:")
print(f"    {qmult_totals}")
print(f"    {qmult_span}")

In [ ]:
# --- Run FLASQ Analysis for Quadratic Multiplier ---
n_total_logical_qubits_qmult = int(10.5 * BITSIZE)
assumptions = frozendict({ROTATION_ERROR: 1e-3, V_CULT_FACTOR: 4.0, T_REACT: 1.0})
qmult_summary_conservative = run_and_print_flasq_analysis(
    "Quadratic Multiplier",
    qmult_cbloq,
    n_total_logical_qubits_qmult,
    conservative_FLASQ_costs,
    "Conservative",
    assumptions,
)
qmult_summary_optimistic = run_and_print_flasq_analysis(
    "Quadratic Multiplier",
    qmult_cbloq,
    n_total_logical_qubits_qmult,
    optimistic_FLASQ_costs,
    "Optimistic",
    assumptions,
)

In [ ]:
# --- 2. Karatsuba Multiplication ---
print("\n2. Analyzing Karatsuba Multiplier...")
# Build the circuit with the specific 2D layout.
kmult_data = build_karatsuba_mult_circuit(bitsize=BITSIZE)

In [ ]:
# Convert the circuit to a CompositeBloq for costing.
kmult_cbloq, _ = convert_circuit_for_flasq_analysis(kmult_data.circuit)

In [ ]:
# Get the abstract gate counts and span costs.
kmult_totals = get_cost_value(kmult_cbloq, FLASQGateTotals())
kmult_span = get_cost_value(kmult_cbloq, TotalSpanCost())

In [ ]:
print("  Resource Counts:")
print(f"    {kmult_totals}")
print(f"    {kmult_span}")

In [ ]:
# --- Run FLASQ Analysis for Karatsuba (Tight Budget) ---
n_total_logical_qubits_kmult_tight = int(4.5 * BITSIZE + 6)
kmult_tight_summary_conservative = run_and_print_flasq_analysis(
    "Karatsuba Multiplier (Tight Budget)",
    kmult_cbloq,
    n_total_logical_qubits_kmult_tight,
    conservative_FLASQ_costs,
    "Conservative",
    assumptions,
)
kmult_tight_summary_optimistic = run_and_print_flasq_analysis(
    "Karatsuba Multiplier (Tight Budget)",
    kmult_cbloq,
    n_total_logical_qubits_kmult_tight,
    optimistic_FLASQ_costs,
    "Optimistic",
    assumptions,
)

In [ ]:
# --- Run FLASQ Analysis for Karatsuba (Generous Budget) ---
n_total_logical_qubits_kmult_generous = int(10 * BITSIZE + 2)
kmult_generous_summary_conservative = run_and_print_flasq_analysis(
    "Karatsuba Multiplier (Generous Budget)",
    kmult_cbloq,
    n_total_logical_qubits_kmult_generous,
    conservative_FLASQ_costs,
    "Conservative",
    assumptions,
)
kmult_generous_summary_optimistic = run_and_print_flasq_analysis(
    "Karatsuba Multiplier (Generous Budget)",
    kmult_cbloq,
    n_total_logical_qubits_kmult_generous,
    optimistic_FLASQ_costs,
    "Optimistic",
    assumptions,
)

In [ ]:
# --- 5. Spacetime Volume Ratios ---
print("\n\n--- Spacetime Volume Ratios (Karatsuba / Quadratic) ---")

In [ ]:
# Conservative Model Ratios
ratio_tight_conservative = (
    kmult_tight_summary_conservative.total_spacetime_volume
    / qmult_summary_conservative.total_spacetime_volume
)
ratio_generous_conservative = (
    kmult_generous_summary_conservative.total_spacetime_volume
    / qmult_summary_conservative.total_spacetime_volume
)
print("\nConservative Model:")
print(f"  Tight Budget Ratio: {ratio_tight_conservative:.2f}x")
print(f"  Generous Budget Ratio: {ratio_generous_conservative:.2f}x")

In [ ]:
# Optimistic Model Ratios
ratio_tight_optimistic = (
    kmult_tight_summary_optimistic.total_spacetime_volume
    / qmult_summary_optimistic.total_spacetime_volume
)
ratio_generous_optimistic = (
    kmult_generous_summary_optimistic.total_spacetime_volume
    / qmult_summary_optimistic.total_spacetime_volume
)
print("\nOptimistic Model:")
print(f"  Tight Budget Ratio: {ratio_tight_optimistic:.2f}x")
print(f"  Generous Budget Ratio: {ratio_generous_optimistic:.2f}x")